# Lesson 4e — `nn.TransformerEncoderLayer` from scratch (runnable)

Every Transformer is a STACK of these. This notebook builds one from five small pieces:

    x ─► LayerNorm ─► MultiHeadAttention ─► add x (residual) ─► h
                                                                 │
        LayerNorm ─► FeedForward ─► add h (residual) ─► output ◄┘

We implement each piece, then verify our composed layer is numerically identical to PyTorch's `nn.TransformerEncoderLayer`.

Runnable version of [`04e_encoder_layer_from_scratch.py`](../04e_encoder_layer_from_scratch.py).


## Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

## Step 1 — LayerNorm from scratch

For each vector: subtract mean, divide by std, apply learned scale + shift.

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))   # Learned scale.
        self.beta  = nn.Parameter(torch.zeros(dim))  # Learned shift.
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        x_normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_normalized + self.beta

x = torch.randn(2, 4, 6)
mine = MyLayerNorm(6)
ref  = nn.LayerNorm(6)
mine.gamma.data.copy_(ref.weight.data)
mine.beta.data.copy_(ref.bias.data)
diff = (mine(x) - ref(x)).abs().max().item()
print(f"Max diff vs nn.LayerNorm: {diff:.2e}")

## Step 2 — Feed-forward sub-layer

Linear → GELU → Linear with 4× expansion (the convention from the original Transformer paper).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_mult=4):
        super().__init__()
        d_hidden = d_model * hidden_mult
        self.up   = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))

ff = FeedForward(6)
print(f"Parameters: {sum(p.numel() for p in ff.parameters())}")
print(f"In: {tuple(x.shape)}  →  Out: {tuple(ff(x).shape)}")

## Step 3 — Multi-head attention (recap from L3c)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, _ = x.shape
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        attn = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.d_k), dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out)

## Step 4 — Put it all together: one encoder layer

Post-norm architecture (PyTorch default): `x → attention → +residual → LayerNorm → FFN → +residual → LayerNorm`.

In [ ]:
class MyTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, dim_feedforward=None):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff   = FeedForward(d_model, hidden_mult=(dim_feedforward // d_model) if dim_feedforward else 4)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))    # Attention + residual + norm.
        x = self.norm2(x + self.ff(x))      # FFN + residual + norm.
        return x

layer = MyTransformerEncoderLayer(d_model=6, n_heads=2, dim_feedforward=24)
print(f"Input shape:  {tuple(x.shape)}")
print(f"Output shape: {tuple(layer(x).shape)}")
print(f"Parameters in one layer: {sum(p.numel() for p in layer.parameters())}")

## Step 5 — Verify against `nn.TransformerEncoderLayer`

In [ ]:
ref = nn.TransformerEncoderLayer(
    d_model=6, nhead=2, dim_feedforward=24,
    activation="gelu", batch_first=True,
    norm_first=False, dropout=0.0,
)
ref.eval()

with torch.no_grad():
    ref.self_attn.in_proj_weight.copy_(torch.cat(
        [layer.attn.W_q.weight, layer.attn.W_k.weight, layer.attn.W_v.weight], dim=0))
    ref.self_attn.in_proj_bias.zero_()
    ref.self_attn.out_proj.weight.copy_(layer.attn.W_o.weight)
    ref.self_attn.out_proj.bias.zero_()
    ref.linear1.weight.copy_(layer.ff.up.weight)
    ref.linear1.bias.copy_(layer.ff.up.bias)
    ref.linear2.weight.copy_(layer.ff.down.weight)
    ref.linear2.bias.copy_(layer.ff.down.bias)
    ref.norm1.weight.copy_(layer.norm1.weight)
    ref.norm1.bias.copy_(layer.norm1.bias)
    ref.norm2.weight.copy_(layer.norm2.weight)
    ref.norm2.bias.copy_(layer.norm2.bias)

out_mine = layer(x)
out_ref  = ref(x)
diff = (out_mine - out_ref).abs().max().item()
print(f"Max absolute difference: {diff:.2e}")
if diff < 1e-5:
    print("✓ Our hand-built encoder layer is numerically identical to PyTorch's.")

## Things to try

1. Try **pre-norm** instead of post-norm: `x = x + self.attn(self.norm1(x))`. Pre-norm trains more stably for deep stacks.
2. Remove the residual connections. Train a deep stack — gradients vanish.
3. Remove the FFN entirely. Model trains but with less capacity.
4. Replace GELU with ReLU. Tiny but consistent difference.